## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-27-20-53-19 +0000,nypost,Nasdaq marks best week in 7 months as tech sto...,https://nypost.com/2026/11/27/business/nasdaq-...
1,2026-11-27-20-51-21 +0000,nyt,Johannesburg Lawmakers Snub Helen Zille in May...,https://www.nytimes.com/2026/11/27/world/afric...
2,2026-11-27-20-50-36 +0000,startribune,Live: Orono takes lead over Byron in third qua...,https://www.startribune.com/minnesota-high-sch...
3,2026-11-27-20-47-01 +0000,wapo,Mexico condemns DEA tip line asking people to ...,https://www.washingtonpost.com/world/2026/11/2...
4,2026-11-27-20-47-00 +0000,wsj,Oil Futures Settle Lower Ahead of OPEC+ Policy...,https://www.wsj.com/finance/commodities-future...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2534/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
116,trump,19
145,new,19
65,thanksgiving,16
150,ai,14
265,out,10


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
236,2026-11-27-04-06-00 +0000,wsj,New Zealand’s election ended in a deadlock aft...,https://www.wsj.com/politics/elections/new-zea...,65
16,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...,56
200,2026-11-27-09-57-50 +0000,cbc,Thanksgiving storm on U.S. East Coast knocks o...,https://www.cbc.ca/news/world/u-s-thanksgiving...,54
151,2026-11-27-12-24-03 +0000,chitri,Ukraine president urges US to pressure Vladimi...,https://www.chicagotribune.com/2026/11/27/ukra...,54
28,2026-11-27-20-00-00 +0000,missionlocal,"Tenderloin Buzz: Three new businesses, just in...",https://missionlocal.org/2026/11/tenderloin-bu...,53


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
236,65,2026-11-27-04-06-00 +0000,wsj,New Zealand’s election ended in a deadlock aft...,https://www.wsj.com/politics/elections/new-zea...
151,54,2026-11-27-12-24-03 +0000,chitri,Ukraine president urges US to pressure Vladimi...,https://www.chicagotribune.com/2026/11/27/ukra...
200,54,2026-11-27-09-57-50 +0000,cbc,Thanksgiving storm on U.S. East Coast knocks o...,https://www.cbc.ca/news/world/u-s-thanksgiving...
100,44,2026-11-27-16-35-08 +0000,nypost,Real cause of death of Deb Fischer’s husband r...,https://nypost.com/2026/11/27/us-news/real-cau...
94,40,2026-11-27-16-53-46 +0000,bbc,Canada's top court largely upholds Quebec ban ...,https://www.bbc.co.uk/news/articles/cq3dgpy9le...
90,35,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...
175,34,2026-11-27-11-00-00 +0000,latimes,"LAUSD blocks Google AI, despite tech giant's m...",https://www.latimes.com/california/story/2026-...
61,31,2026-11-27-18-27-44 +0000,nypost,NYC’s oldest and best small businesses need yo...,https://nypost.com/2026/11/27/lifestyle/2026-s...
240,30,2026-11-27-03-00-00 +0000,wsj,"No Homes to Rent, No Way to Evict: Inside Spai...",https://www.wsj.com/world/europe/spain-housing...
134,30,2026-11-27-13-39-00 +0000,wsj,The French economy tipped into a recession in ...,https://www.wsj.com/world/europe/french-econom...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
